# Speaker verification and diarization with ESPnet-SPK

Extract speaker embeddings with a pretrained ESPnet-SPK model, verify whether
two utterances come from the same speaker, and run a simple diarization of a
two-speaker mixture.

This is the demonstration part of an assignment from CMU 11492/11692/18495,
*Speech Technology for Conversational AI*, kept here without the graded
exercises so that it runs end to end.

Main references:
- [ESPnet repository](https://github.com/espnet/espnet)
- [ESPnet documentation](https://espnet.github.io/espnet/)
- [ESPnet-SPK recipe template](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/spk1)
- [ESPnet-SPK paper](https://arxiv.org/abs/2401.17230)

Author:
- Chyi-Jiunn Lin (chyijiul@andrew.cmu.edu)

## Acknowledgement
- Adapted from Jiatong Shi's version in 11692 (jiatongs@andrew.cmu.edu)


# Contents

1. Install
2. Speaker embedding extraction
3. Speaker embedding visualization
4. A speaker verification system
5. Speaker diarization


## Install

We use the inference install of ESPnet rather than the full one: it carries no
training stack, so it is much smaller and much faster to install.


In [ ]:
import locale
locale.getpreferredencoding = lambda *a, **kw: "UTF-8"

%pip install -q "espnet==202610.post1"
!pip install ffmpeg-python

In [ ]:
!pip install librosa
!pip install transformers


## Speaker Embedding Extraction


Use pre-trained speaker recognition model to extract speaker embedding.


In [ ]:
#@title Choose speaker recognition model { run: "auto" }
input_feature = "mel" #@param ["mel", "frozen_wavlm", "wavlm", "none"]
architecture = "ecapa" #@param ["ecapa", "xvector", "rawnet"]


if architecture == "rawnet":
  assert input_feature == "none"
  model_tag = "espnet/voxcelebs12_rawnet3"
elif architecture == "xvector":
  assert input_feature == "mel"
  model_tag = "espnet/voxcelebs12_xvector_mel"
else:
  if input_feature == "mel":
    model_tag = "espnet/voxcelebs12_ecapa_mel"
  elif input_feature == "wavlm":
    model_tag = "espnet/voxcelebs12_ecapa_wavlm_joint"
  elif input_feature == "frozen_wavlm":
    model_tag = "espnet/voxcelebs12_ecapa_frozen"
  else:
    raise ValueError("ECAPA TDNN cannot use none features")

# Only the WavLM front-ends need S3PRL, so only they install it: it is a
# large package and the other three choices never import it.
#
# Those two choices do not currently work. S3PRL's released version calls
# torchaudio.set_audio_backend, removed in torchaudio 2.1, and espnet pins
# 2.11 - so the model loads far enough to raise AttributeError. Pick "mel"
# or "none" until that is fixed upstream.
if input_feature in ("wavlm", "frozen_wavlm"):
  !pip install s3prl

For this demonstration, we use a small subset of the Librispeech dataset, namely [mini-librispeech](https://www.openslr.org/31/). We first download the data to the google drive through `wget`.

The layout of the dataset is as
```
- LibriSpeech
    - dev-clean-2
       - 1272 (spk1_id)
           - 135031 (book1_id)
               - 1272-135031-0000.flac (speaker audios)
               - 1272-135031-0001.flac
               - ...
           - 141231 (book2_id)
               - ...
       - 1462 (spk2_id)
           - ...
       ...
```


In [ ]:
# mini-librispeech: five speakers, a few minutes of read speech

!wget https://us.openslr.org/resources/31/dev-clean-2.tar.gz --no-check-certificate
!tar -xzvf dev-clean-2.tar.gz

### Task1  )

Conduct speaker embedding extraction based on the selected model.

Please pick up five speakers from the speaker inside the dataset and fill them in the python code (line). We will use their corresponding utterances to extract speaker embedding.


In [ ]:
import soundfile
from IPython.display import display, Audio

SPEAKERS = ["1272", "174", "251", "84", "777"] # fill in the spk id from the dataset
              # an example would be ["1272", "1462", "251", "84", "777"]

# A dict that records the speaker-wave information
speaker2wav = {spk: [] for spk in SPEAKERS}

# Get the audio paths for selected speaker
import os
base_dir = "LibriSpeech/dev-clean-2"
for spk in SPEAKERS:
  for book in os.listdir(os.path.join(base_dir, spk)):
    for audio_path in os.listdir(os.path.join(base_dir, spk, book)):
      if audio_path.endswith("flac"):
        speaker2wav[spk].append(os.path.join(base_dir, spk, book, audio_path))
    # only the first book: the verification utterance later comes
    # from another one, and it must not be in the enrollment average
    break

# Present the first speech for each selected speaker
for spk in SPEAKERS:
  print("Speaker {}".format(spk))
  audio, sr = soundfile.read(speaker2wav[spk][0])
  print(audio.shape)
  display(Audio(audio, rate=sr))


We then extract the speaker embedding with the pre-trained speaker recognition model.

The procedure is very simple with the ESPnet-SPK toolkit: (1) initialize the model; (2) use the model to extract speaker embedding.

Following many previous works, we also average the speaker embedding of the same speaker.


In [ ]:
import numpy as np
import torch
from espnet2.bin.spk_inference import Speech2Embedding

speech2embedding = Speech2Embedding.from_pretrained(
    model_tag=model_tag,
    device="cuda" if torch.cuda.is_available() else "cpu" if torch.cuda.is_available() else "cpu",
)

In [ ]:
from tqdm import tqdm

speaker2embeds = {spk: [] for spk in SPEAKERS}
speaker2avg_embed = {}

for spk in SPEAKERS:
  print("processing speaker {}".format(spk))
  for audio_path in tqdm(speaker2wav[spk]):
    audio, sr = soundfile.read(audio_path)
    embedding = speech2embedding(audio).squeeze(0).cpu().numpy()
    speaker2embeds[spk].append(embedding)
  speaker2avg_embed[spk] = np.average(np.stack(speaker2embeds[spk]), axis=0)
  print(speaker2avg_embed[spk].shape)


### Task2  )

Visualization is an important step to ensure our speaker embedding quality. However, high-dimensional vector is difficult to check. Usually, we use some dimension reduction techniques to project the vector to 2D form for better understanding of the extracted speaker embedding.

You can use the following plot script to visualize the speaker embeddings of the five selcted speakers.


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

speaker_keys = []
speaker_values = []
for spk in SPEAKERS:
  embeddings = speaker2embeds[spk]
  for embedding in embeddings:
    speaker_keys.append(spk)
    speaker_values.append(embedding)

# Apply t-SNE transformation
tsne = TSNE(n_components=2, random_state=0)
embeddings_2d = tsne.fit_transform(np.array(speaker_values))

# Plotting
plt.figure()
colors = plt.cm.rainbow(np.linspace(0, 1, len(SPEAKERS)))
speaker2color = {spk: colors[i] for i, spk in enumerate(SPEAKERS)}
legends = []
for j in range(len(embeddings_2d)):
  spk = speaker_keys[j]
  if spk not in legends:
    plt.scatter(embeddings_2d[j, 0], embeddings_2d[j, 1], color=speaker2color[spk], label=spk)
    legends.append(spk)
  else:
    plt.scatter(embeddings_2d[j, 0], embeddings_2d[j, 1], color=speaker2color[spk])

plt.legend()
plt.xlabel('t-SNE feature 0')
plt.ylabel('t-SNE feature 1')
plt.title('t-SNE visualization of Speaker Embeddings')
plt.show()

### Task3  )

Speaker verification is a typical task by using the speaker embedding. In this demonstration, we show how to use the enrolled speaker embedding to conduct speaker verification.

There are two cases for the speaker verification system: accept and reject. This time, we showcase a simple way by setting a threshold of the cosine similarity between the embedding of the input voice and the embedding of the enrollment speech.

Please select one speaker in the selected five speaker in mini-librispeech as your enrolled speaker (we will use the average embedding as his/her speaker embedding). Then test it with his/her own voices by providing one of his/her speech file path.

To make the test challenging, please consider selecting utterances from the other book then the selected one.


In [ ]:
ENROLL_SPEAKER = "251" # choose one of the five speaker embedding you selected for the above tasks.
THRESHOLD = 0.5

enroll_embedding = speaker2avg_embed[ENROLL_SPEAKER]
test_utt_path = "LibriSpeech/dev-clean-2/251/136532/251-136532-0000.flac"

# Calculate embedding for the new utterance
audio, sr = soundfile.read(test_utt_path)
test_embedding = speech2embedding(audio).squeeze().cpu().numpy()

# Calculate cosine similarity
cosine_sim = np.dot(enroll_embedding, test_embedding) / (np.linalg.norm(enroll_embedding) * np.linalg.norm(test_embedding))

# Use threshold to conduct speaker verification
if cosine_sim > THRESHOLD:
  print("ACCEPT with utterance {} for speaker {} with similarity {}".format(test_utt_path, ENROLL_SPEAKER, cosine_sim))
else:
  print("REJECT with utterance {} for speaker {} with similarity {}".format(test_utt_path, ENROLL_SPEAKER, cosine_sim))


### Task4  )

Now, use the same code but select utterances from another speaker to test the robustness of your system.


In [ ]:
test_utt_path = "LibriSpeech/dev-clean-2/3000/15664/3000-15664-0000.flac" # provide one utterance for test

# Calculate embedding for the new utterance
audio, sr = soundfile.read(test_utt_path)
test_embedding = speech2embedding(audio).squeeze().cpu().numpy()

# Calculate cosine similarity
cosine_sim = np.dot(enroll_embedding, test_embedding) / (np.linalg.norm(enroll_embedding) * np.linalg.norm(test_embedding))

# Use threshold to conduct speaker verification
if cosine_sim > THRESHOLD:
  print("ACCEPT with utterance {} for speaker {} with similarity {}".format(test_utt_path, ENROLL_SPEAKER, cosine_sim))
else:
  print("REJECT with utterance {} for speaker {} with similarity {}".format(test_utt_path, ENROLL_SPEAKER, cosine_sim))

## Speaker Diarization

### Task 5  )

In this section, we demonstrate **speaker diarization**: given a multi-speaker audio stream, determine *who spoke when*.

The pipeline is:
1. **Mix** — concatenate audio from two enrolled speakers to create a 20-second mixture (0–10 s: speaker A, 10–20 s: speaker B)
2. **Segment** — slice the mixture into non-overlapping 5-second windows
3. **Embed** — extract a speaker embedding for each segment using `speech2embedding`
4. **Assign** — compare each segment embedding to enrolled speaker embeddings in `speaker2avg_embed` via cosine similarity and label accordingly


First, select the diarization mode and randomly pick two enrolled speakers to mix.


In [ ]:
import random

DIARIZATION_MODE = "nearest"  # "threshold": reject segment if best similarity < THRESHOLD
                                 # "nearest":   always assign to the most similar enrolled speaker
WINDOW_SIZE = 5  # seconds, size of each analysis window
STRIDE = 5       # seconds, step between windows (set STRIDE < WINDOW_SIZE for overlapping windows)

spk_a, spk_b = random.sample(SPEAKERS, 2)
print(f"Mixing speakers: {spk_a} and {spk_b}")

Concatenate utterances from each speaker to build a 20-second mixture (0–10 s: speaker A, 10–20 s: speaker B). The ground truth label for each segment is determined by which speaker covers the majority of that window.


In [ ]:
from IPython.display import Audio
import librosa

SR = 16000
TARGET_DURATION = 10  # seconds per speaker
target_samples = TARGET_DURATION * SR  # 160000 samples

def collect_audio(spk, n_samples):
    """Concatenate utterances from spk until n_samples is reached, then truncate."""
    collected = []
    total = 0
    for path in speaker2wav[spk]:
        audio, sr = soundfile.read(path)
        audio = librosa.resample(audio.astype(np.float32), orig_sr=sr, target_sr=SR)
        collected.append(audio)
        total += len(audio)
        if total >= n_samples:
            break
    combined = np.concatenate(collected)
    return combined[:n_samples]

audio_a = collect_audio(spk_a, target_samples)
audio_b = collect_audio(spk_b, target_samples)

mixture = np.concatenate([audio_a, audio_b])
total_duration = len(mixture) // SR  # 20 seconds

# Ground truth: assign each segment to the speaker covering the majority of its window
ground_truth = []
for start in range(0, total_duration - WINDOW_SIZE + 1, STRIDE):
    seg_end = start + WINDOW_SIZE
    spk_a_coverage = max(0, min(seg_end, TARGET_DURATION) - start)
    spk_b_coverage = max(0, seg_end - max(start, TARGET_DURATION))
    ground_truth.append(spk_a if spk_a_coverage >= spk_b_coverage else spk_b)

print(f"Mixture duration: {total_duration}s  ({spk_a}: 0–{TARGET_DURATION}s, {spk_b}: {TARGET_DURATION}–{total_duration}s)")
print(f"Ground truth per segment: {ground_truth}")
display(Audio(mixture, rate=SR))

Slice the mixture into fixed-length windows and extract a speaker embedding for each segment using the pre-trained `speech2embedding` model.


In [ ]:
segments = []
for start in range(0, total_duration - WINDOW_SIZE + 1, STRIDE):
    seg_audio = mixture[start * SR : (start + WINDOW_SIZE) * SR]
    embedding = speech2embedding(seg_audio).squeeze(0).cpu().numpy()
    segments.append({
        "start": start,
        "end": start + WINDOW_SIZE,
        "embedding": embedding,
    })
    print(f"Extracted segment [{start}s–{start + WINDOW_SIZE}s], embedding shape: {embedding.shape}")

print(f"\nTotal segments: {len(segments)}")

Compare each segment's embedding to the enrolled speaker embeddings in `speaker2avg_embed` using cosine similarity. Assign a speaker label based on the selected mode: in `threshold` mode, segments below the similarity threshold are labeled `"unknown"`, and in `nearest` mode, the closest speaker is always assigned.


In [ ]:
def compute_cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

for i, seg in enumerate(segments):
    sims = {spk: compute_cosine_sim(seg["embedding"], speaker2avg_embed[spk]) for spk in SPEAKERS}
    best_spk = max(sims, key=sims.get)
    best_sim = sims[best_spk]

    if DIARIZATION_MODE == "threshold":
        label = best_spk if best_sim > THRESHOLD else "unknown"
    else:
        label = best_spk

    seg["label"] = label
    seg["similarity"] = best_sim

    gt = ground_truth[i]
    correct = "✓" if label == gt else "✗"
    print(f"[{seg['start']}s–{seg['end']}s]  → Speaker {label:<6}  (similarity: {best_sim:.3f})  [GT: {gt}] {correct}")

Calculate the Diarization Error Rate (DER), which measures the fraction of audio time that was incorrectly labeled.


In [ ]:
from collections import Counter

# Note:
# Here we evaluate DER at 1-second resolution using majority vote over overlapping segments.
# For non-overlapping windows (STRIDE == WINDOW_SIZE), each second belongs to exactly one segment.
# For overlapping windows (STRIDE < WINDOW_SIZE), multiple segments cover each second;
# we take a majority vote among their labels.
incorrect_time = 0.0

for t in range(total_duration):
    covering = [seg for seg in segments if seg["start"] <= t < seg["end"]]
    if not covering:
        continue
    vote = Counter(seg["label"] for seg in covering).most_common(1)[0][0]
    gt_t = spk_a if t < TARGET_DURATION else spk_b
    if vote != gt_t:
        incorrect_time += 1.0

der = incorrect_time / total_duration * 100
print(f"Diarization Error Rate (DER): {der:.1f}%")
print(f"  Incorrect predictions: {incorrect_time:.0f}s  /  Total: {total_duration}s")

Visualize the diarization results as a timeline. Each bar represents a stride-length window, color-coded by speaker. The top row shows the model's predictions, while the bottom row shows the ground truth.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Extend the existing speaker2color palette with a gray for "unknown"
diarization_colors = dict(speaker2color)
diarization_colors["unknown"] = (0.6, 0.6, 0.6, 1.0)

fig, ax = plt.subplots(figsize=(12, 2.5))

rows = {
    "Predicted":    [seg["label"] for seg in segments],
    "Ground Truth": ground_truth,
}
y_positions = {"Predicted": 1, "Ground Truth": 0}

for row_name, labels in rows.items():
    y = y_positions[row_name]
    for i, label in enumerate(labels):
        start = segments[i]["start"]
        width = STRIDE  # each bar represents one stride-length prediction window
        color = diarization_colors.get(label, (0.6, 0.6, 0.6, 1.0))
        ax.barh(y, width, left=start, height=0.6, color=color, edgecolor="white", linewidth=1.5)
        if STRIDE >= 2:  # only draw text label if bar is wide enough to read
            ax.text(start + width / 2, y, label, ha="center", va="center",
                    fontsize=9, color="white", fontweight="bold")

ax.set_yticks([0, 1])
ax.set_yticklabels(["Ground Truth", "Predicted"])
ax.set_xlabel("Time (s)")
ax.set_xticks(range(0, total_duration + 1, 5))
ax.set_xlim(0, total_duration)
ax.set_title(f"Speaker Diarization Timeline  (DER: {der:.1f}%, mode: {DIARIZATION_MODE}, window={WINDOW_SIZE}s, stride={STRIDE}s)")

legend_labels = SPEAKERS + (["unknown"] if DIARIZATION_MODE == "threshold" else [])
patches = [mpatches.Patch(color=diarization_colors[s], label=f"Speaker {s}")
           for s in legend_labels if s in diarization_colors]
ax.legend(handles=patches, loc="upper right", bbox_to_anchor=(1.18, 1.1))

plt.tight_layout()
plt.show()